# 02_2 — Gộp thiết bị field-capture vào dataset train

`Data/data_pcap/` là các lần bắt gói trên cả mạng nhà (điện thoại, laptop, PC,
Raspberry Pi...), nhiều thiết bị trộn chung — khác `POWER` (1 pcap = 1 thiết bị = 1 sự
kiện ngắn). Vận hành thực tế bắt bằng cách restart tcpdump mỗi vài phút (mỗi lần chạy
`02_2` lại thấy nhiều file `.pcap` rời nhau, không phải một file liên tục nhiều giờ), nên
notebook này **windowing theo file pcap**: mỗi `(canonical_device, capture_file)` là một
session (`session_id = f"{canonical_device}_{Path(capture_file).stem}"`) — khớp đúng ranh
giới bằng chứng thật đã có, thay vì gộp theo giờ đồng hồ như IDLE (giả định đó chỉ đúng
khi capture chạy liên tục nhiều giờ không ngắt quãng, không phải trường hợp ở đây — gộp
theo giờ từng nén 4-6 file cùng một khung giờ thành 1 session, làm rớt hẳn số session mỗi
thiết bị có xuống dưới ngưỡng `min_windows` ở `06_test_model.ipynb`).

Notebook **không đọc lại pcap lần hai** — chạy `01_2_new_capture_features.ipynb` qua
`%run -i` để lấy `records_by_session` (đã bóc DHCP/DNS/mDNS/TLS) đang có sẵn trong bộ nhớ,
rồi gom lại theo `(canonical_device, date, hour)` thay vì `(mac, capture_file)`.

Các bước:
1. Chạy `01_2` lấy record thô + `mac_to_device`.
2. Gán nhãn ground-truth (make/type/model) cho từng hostname đã biết trong
   `Data/data_pcap/device_map.csv` — MAC không có hostname thật (`Unknown-*`) bị loại,
   không đưa vào train.
3. Windowing theo giờ, gọi `aggregate()` (dùng lại từ `03_train_model.ipynb`) — đúng cột,
   đúng quy tắc như dữ liệu CIC.
4. Gộp vào `Data/sessions.parquet` (raw, ghi đè các session cũ có `scenario == "FIELD"`
   để chạy lại không bị nhân đôi).
5. Thêm dòng vào `Data/device_labels_verified.csv` (chỉ thiết bị thực sự có session).
6. Gọi `build_verified_dataset()` ngay tại đây để dựng `Data/sessions_verified.parquet` —
   không cần bật `REBUILD_VERIFIED` bên `03_train_model.ipynb` nữa, cứ mở `03` **Run All**
   để train.


In [1]:
# Mô tả: Cấu hình biến môi trường và số luồng cho BLAS/TF
import os

os.environ['OPENBLAS_NUM_THREADS'] = '44'  # 50% cores
os.environ['MKL_NUM_THREADS'] = '44'
os.environ['OMP_NUM_THREADS'] = '44'
os.environ['NUMEXPR_NUM_THREADS'] = '44'

# TensorFlow threading
os.environ['TF_NUM_INTRAOP_THREADS'] = '44'  # Parallel ops
os.environ['TF_NUM_INTEROP_THREADS'] = '8'   # Independent ops

# turn off oneDNN optimization if needed
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print("Configured for 88-core CPU")

Configured for 88-core CPU


In [2]:
# Nạp định nghĩa từ 03 mà không chạy train — cùng cách 01/01_1/01_2/06 vẫn làm.
from pathlib import Path

_cwd = Path.cwd().resolve()
_ROOT = next((p for p in (_cwd, *_cwd.parents)
              if (p / "Code" / "03_train_model.ipynb").is_file()), None)
assert _ROOT is not None, f"Không thấy Code/03_train_model.ipynb quanh {_cwd}"
_CODE = _ROOT / "Code"

if not globals().get("SDC_DEFS_LOADED"):
    SDC_IMPORT_ONLY = True
    try:
        get_ipython().run_line_magic("run", f'-i "{_CODE / "03_train_model.ipynb"}"')
    finally:
        del SDC_IMPORT_ONLY

import pandas as pd
from IPython.display import display


Configured for 88-core CPU
Gốc dự án: /home/ubuntu/sepcung/02.SDC
Đã nạp hàm SDC từ 03_train_model.ipynb


## 1. Chạy `01_2_new_capture_features.ipynb`

Lấy `records_by_session`, `mac_to_device`, `features` — không viết lại parser. Notebook
con vẫn ghi `*_capture.csv` như bình thường (dùng để chấm Predictor), ta chỉ mượn dữ liệu
trong bộ nhớ sau khi nó chạy xong.


In [3]:
get_ipython().run_line_magic("run", f'-i "{_CODE / "01_2_new_capture_features.ipynb"}"')

print(f"\n{len(records_by_session)} (mac, capture_file) từ {len(pcap_files)} file pcap")


/home/ubuntu/thangdh/python_env/lib/python3.10/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)
/home/ubuntu/thangdh/python_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


scapy  : 2.7.0
6 file pcap
  sdc_pcap_14092026_16h43.pcap  (4005.4 MB)
  sdc_pcap_14092026_16h46.pcap  (1278.4 MB)
  sdc_pcap_14092026_16h48.pcap  (1620.5 MB)
  sdc_pcap_14092026_16h58.pcap  (3827.4 MB)
  sdc_pcap_14092026_17h03.pcap  (2565.3 MB)
  sdc_pcap_14092026_17h14.pcap  (1504.6 MB)
15 thiet bi trong device_map.csv


capture files: 100%|██████████| 6/6 [48:19<00:00, 483.21s/it]  


dhcp_df: (41, 13) dns_df: (1667, 12) mdns_df: (976, 12) tls_df: (1428, 14)
15 MAC nguồn, 62 session (mac, capture_file)
Đã lưu vào /home/ubuntu/sepcung/02.SDC/Data/features
 - dhcp_features_capture.csv
 - dns_features_capture.csv
 - mdns_features_capture.csv
 - tls_features_capture.csv
 - capture_summary_capture.csv
62 session x 40 cột feature
Đã lưu /home/ubuntu/sepcung/02.SDC/Data/features/session_features_capture.csv


,mac,device,capture_file,has_dhcp,has_dns,has_mdns,has_tls,n_sources,dhcp_vci,tls_sni_tokens
0,e0:8f:4c:90:33:96,DESKTOP-cua-Lee-KingDom,sdc_pcap_14092026_16h43.pcap,0,1,0,1,2,<missing>,settings win data microsoft com ecs office log...
1,16:76:ae:9b:f1:a0,Unknown-1676ae9bf1a0,sdc_pcap_14092026_16h43.pcap,0,1,1,1,3,<missing>,valid apple com ocsp2 imap gmail mask api iclo...
2,00:e0:4c:53:44:58,linova,sdc_pcap_14092026_16h43.pcap,0,1,0,1,2,<missing>,chatgpt com teams events data microsoft vntek ...
3,e0:d0:45:b0:da:e7,DESKTOP-DucAnh,sdc_pcap_14092026_16h43.pcap,0,1,0,1,2,<missing>,label wpa chat zalo me vscode sync trafficmana...
4,50:bb:b5:fe:2e:d0,LAPTOP-VAF70SQ6,sdc_pcap_14092026_16h43.pcap,0,1,0,1,2,<missing>,e2c83 gcp gvt2 com chat zalo me browser intake...
...,...,...,...,...,...,...,...,...,...,...
57,e0:d0:45:b0:da:e7,DESKTOP-DucAnh,sdc_pcap_14092026_17h14.pcap,1,1,1,1,4,MSFT 5.0,dns google pub ent auea 06 t trouter teams mic...
58,e4:5f:01:a3:c6:0d,raspberrypi,sdc_pcap_14092026_17h14.pcap,1,1,1,1,4,<missing>,push services mozilla com merino safebrowsing ...
59,5a:e8:f1:fc:4d:ba,Samsung-cua-Duc-Anh,sdc_pcap_14092026_17h14.pcap,1,1,1,1,4,android-dhcp-16,www google com mtalk z m gateway facebook qos ...
60,4c:12:e8:40:01:e7,IP-Camera,sdc_pcap_14092026_17h14.pcap,0,1,0,0,1,<missing>,<missing>


Phân bố số nguồn feature có mặt / (mac, capture_file):


n_sources
1    11
2    25
3    10
4    16
Name: count, dtype: int64


Chi tiết theo MAC:


,,has_dhcp,has_dns,has_mdns,has_tls
mac,device,,,,
00:e0:4c:53:44:58,linova,1,1,0,1
06:27:04:ee:2a:40,Unknown-062704ee2a40,1,1,1,1
16:76:ae:9b:f1:a0,Unknown-1676ae9bf1a0,1,1,1,1
4c:12:e8:40:01:e7,IP-Camera,0,1,0,0
50:bb:b5:fc:5b:c2,DESKTOP-DNGJHRT,1,1,1,1
50:bb:b5:fe:2e:d0,LAPTOP-VAF70SQ6,0,1,1,1
5a:e8:f1:fc:4d:ba,Samsung-cua-Duc-Anh,1,1,1,1
5a:fe:7e:36:89:76,OPPO-A92,1,1,1,1
6a:13:aa:1f:1e:85,Redmi-Note-10,0,0,1,1



62 (mac, capture_file) từ 6 file pcap


## 2. Nhãn ground-truth cho thiết bị field-capture

`Data/data_pcap/device_labels_capture.csv` (hostname, canonical_device, make, type,
model) — sửa trực tiếp file CSV này khi thêm/đổi thiết bị trong `device_map.csv`, không
cần sửa notebook. MAC không có hostname thật (`Unknown-*`) bị loại thẳng, không vào
dataset train. Hostname có trong `device_map.csv` nhưng chưa có dòng nhãn tương ứng sẽ
chỉ bị cảnh báo và bỏ qua ở mục 3, không làm hỏng cả lần chạy.


In [4]:
import csv

CAPTURE_SCENARIO = "FIELD"
CAPTURE_LABELS_CSV = ROOT / "Data" / "data_pcap" / "device_labels_capture.csv"

with open(CAPTURE_LABELS_CSV, newline="", encoding="utf-8-sig") as f:
    CAPTURE_LABELS = {
        row["hostname"].strip(): {
            "canonical_device": row["canonical_device"].strip(),
            "make": row["make"].strip(),
            "type": row["type"].strip(),
            "model": row["model"].strip(),
        }
        for row in csv.DictReader(f)
    }

print(f"{len(CAPTURE_LABELS)} thiết bị có nhãn trong {CAPTURE_LABELS_CSV.name}")

known_hosts = set(mac_to_device.values())
skipped = {h for h in known_hosts if h.startswith("Unknown-")}
missing_labels = known_hosts - skipped - set(CAPTURE_LABELS)
print(f"Bỏ qua (không rõ hostname): {sorted(skipped) if skipped else 'không có'}")
if missing_labels:
    print(f"CẢNH BÁO: có hostname trong device_map.csv nhưng CHƯA có trong {CAPTURE_LABELS_CSV.name}: "
          f"{sorted(missing_labels)} — thêm dòng cho chúng vào CSV nếu muốn đưa vào train")


12 thiết bị có nhãn trong device_labels_capture.csv
Bỏ qua (không rõ hostname): ['Unknown-062704ee2a40', 'Unknown-1676ae9bf1a0', 'Unknown-6e4c77fc06d3']


## 3. Windowing theo file pcap

Gom lại record thô đã có trong `records_by_session` (khoá `(mac, capture_file)`) thành
khoá `(canonical_device, capture_file)` — 1 session = 1 file pcap có traffic của 1 thiết
bị. Đây chính là ranh giới capture thật (mỗi lần restart tcpdump), không phải khung giờ
đồng hồ như quy ước IDLE — dùng khung giờ ở đây sẽ nén nhiều file capture rời nhau của
cùng một giờ lại thành 1 session, làm mất granularity đã có sẵn trong dữ liệu.


In [5]:
from collections import defaultdict
from datetime import datetime, timezone

windowed = defaultdict(list)          # (canonical_device, capture_file) -> [record, ...]
window_mac = {}                        # (canonical_device, capture_file) -> mac
window_date = {}                       # (canonical_device, capture_file) -> date (ngày của record đầu)
unlabeled_macs = set()

for (mac, capture_file), records in records_by_session.items():
    hostname = mac_to_device.get(mac)
    if hostname is None or hostname.startswith("Unknown-"):
        continue
    label = CAPTURE_LABELS.get(hostname)
    if label is None:
        unlabeled_macs.add(hostname)
        continue
    canonical_device = label["canonical_device"]
    key = (canonical_device, capture_file)
    for record in records:
        ts = record.get("ts")
        if ts is None:
            continue
        windowed[key].append(record)
        window_mac[key] = mac
        if key not in window_date:
            window_date[key] = datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%Y-%m-%d")

if unlabeled_macs:
    print(f"CẢNH BÁO: hostname có trong device_map.csv nhưng thiếu trong CAPTURE_LABELS: {sorted(unlabeled_macs)}")

print(f"{len(windowed)} session (device, file pcap) từ {sum(len(v) for v in windowed.values())} record")


50 session (device, file pcap) từ 3223 record


## 4. `aggregate()` từng session — đúng hàm `03_train_model.ipynb` dùng lúc train/serve

`features` đã có sẵn từ bước chạy `01_2` ở mục 1 (nó tự gọi `load_sessions(SESSIONS_PATH)`
rồi lấy `feature_groups["all"]`).


In [6]:
from pathlib import Path

DATA = ROOT / "Data"
RAW_SESSIONS_PATH = DATA / "sessions.parquet"

agg_rows = []
for key, records in windowed.items():
    canonical_device, capture_file = key
    hostname = next(h for h, l in CAPTURE_LABELS.items() if l["canonical_device"] == canonical_device)
    label = CAPTURE_LABELS[hostname]
    row = {
        "session_id": f"{canonical_device}_{Path(capture_file).stem}",
        "canonical_device": canonical_device,
        "scenario": CAPTURE_SCENARIO,
        "capture_file": capture_file,
        "date": window_date[key],
        "mac": window_mac[key],
        **aggregate(records, features),
        "make": label["make"],
        "type": label["type"],
        "family": label["model"],  # cột thô trong sessions.parquet vẫn tên "family"
    }
    agg_rows.append(row)

new_sessions = pd.DataFrame(agg_rows).set_index("session_id").sort_index()
print(f"{len(new_sessions)} session mới — {new_sessions.canonical_device.nunique()} thiết bị")
display(new_sessions.groupby("canonical_device").size().rename("n_session"))


50 session mới — 12 thiết bị


canonical_device
Desktop PC DNGJHRT          5
Desktop PC DucAnh           6
IP Camera (field)           6
Laptop VAF70SQ6             6
Lee Kingdom Laptop          5
Linova Laptop (linova)      6
OPPO A92                    1
Raspberry Pi                3
Samsung Phone (Duc Anh)     6
Xiaomi Redmi Note 10        1
Xiaomi Redmi Note 14 Pro    2
iPhone                      3
Name: n_session, dtype: int64

## 5. Gộp vào `Data/sessions.parquet` (raw)

Ghi đè các session `scenario == "FIELD"` cũ (nếu có, từ lần chạy trước) để chạy lại
notebook này bao nhiêu lần cũng không bị nhân đôi dữ liệu.


In [7]:
raw = pd.read_parquet(RAW_SESSIONS_PATH)

missing_cols = set(new_sessions.columns) - set(raw.columns)
extra_cols = set(raw.columns) - set(new_sessions.columns)
assert not missing_cols and not extra_cols, (
    f"Lệch cột so với {RAW_SESSIONS_PATH.name}: thiếu {missing_cols}, thừa {extra_cols}"
)

raw = raw[raw["scenario"] != CAPTURE_SCENARIO]
combined = pd.concat([raw, new_sessions[raw.columns]])
combined = combined[~combined.index.duplicated(keep="last")].sort_index()
combined.to_parquet(RAW_SESSIONS_PATH)

print(f"Đã ghi {RAW_SESSIONS_PATH}: {len(combined)} session "
      f"({len(new_sessions)} FIELD mới, {combined.canonical_device.nunique()} thiết bị)")


Đã ghi /home/ubuntu/sepcung/02.SDC/Data/sessions.parquet: 8373 session (50 FIELD mới, 52 thiết bị)


## 6. Cập nhật `Data/device_labels_verified.csv`

Chỉ thêm thiết bị **thực sự có session** ở bước 4 — nếu một thiết bị trong
`CAPTURE_LABELS` không bắt được gói DHCP/DNS/mDNS/TLS nào trong toàn bộ 7 file pcap thì
`build_verified_dataset()` ở mục 7 sẽ báo lỗi "devices absent from sessions" nếu ta lỡ
thêm nhãn cho nó — nên lọc trước ở đây.


In [8]:
LABELS_PATH = DATA / "device_labels_verified.csv"
labels_df = pd.read_csv(LABELS_PATH)

present_devices = set(new_sessions["canonical_device"].unique())
new_label_rows = []
for hostname, label in CAPTURE_LABELS.items():
    cd = label["canonical_device"]
    if cd not in present_devices:
        print(f"CẢNH BÁO: {cd} ({hostname}) không có session nào ở lần chạy này — KHÔNG thêm vào taxonomy")
        continue
    new_label_rows.append({
        "canonical_device": cd,
        "make": label["make"],
        "type": label["type"],
        "model": label["model"],
        "review_note": f"field capture — {hostname} ({CAPTURE_SCENARIO})",
    })

added = {row["canonical_device"] for row in new_label_rows}
labels_df = labels_df[~labels_df["canonical_device"].isin(added)]
labels_df = pd.concat([labels_df, pd.DataFrame(new_label_rows)], ignore_index=True)
labels_df.to_csv(LABELS_PATH, index=False)

print(f"Đã ghi {LABELS_PATH}: +{len(new_label_rows)} thiết bị field-capture")


Đã ghi /home/ubuntu/sepcung/02.SDC/Data/device_labels_verified.csv: +12 thiết bị field-capture


## 7. Dựng lại `Data/sessions_verified.parquet` ngay tại đây

Gọi thẳng `build_verified_dataset()` (hàm dùng chung của `03_train_model.ipynb`) thay vì
đợi bật `REBUILD_VERIFIED` bên đó — lỗi taxonomy (thiết bị chưa phân loại, nhãn trống,
`model` trỏ hai `make` khác nhau...) hiện ngay ở đây thay vì lúc train.


In [9]:
EXCLUDED_PATH = DATA / "device_labels_excluded.csv"

result = build_verified_dataset(
    source=RAW_SESSIONS_PATH,
    labels_path=LABELS_PATH,
    excluded_path=EXCLUDED_PATH,
    output=SESSIONS_PATH,
)

print(f"\n{result.canonical_device.nunique()} thiết bị trong {SESSIONS_PATH.name} sau khi gộp")
display(result[result.scenario == CAPTURE_SCENARIO].groupby("canonical_device").size().rename("n_session"))


wrote /home/ubuntu/sepcung/02.SDC/Data/sessions_verified.parquet: 8189 sessions, 51 devices; excluded=['Smart Board']

51 thiết bị trong sessions_verified.parquet sau khi gộp


canonical_device
Desktop PC DNGJHRT          5
Desktop PC DucAnh           6
IP Camera (field)           6
Laptop VAF70SQ6             6
Lee Kingdom Laptop          5
Linova Laptop (linova)      6
OPPO A92                    1
Raspberry Pi                3
Samsung Phone (Duc Anh)     6
Xiaomi Redmi Note 10        1
Xiaomi Redmi Note 14 Pro    2
iPhone                      3
Name: n_session, dtype: int64

## Xong

`Data/sessions_verified.parquet` đã có sẵn các thiết bị field-capture. Mở
`03_train_model.ipynb` và **Run All** để train — **không cần** bật `REBUILD_VERIFIED`,
dataset đã được dựng lại ở mục 7.
